In [3]:
import os
import sys
import torch
from thop import profile

# ==========================================
# 1. 環境路徑設定
# ==========================================
project_segformer_path = r"C:\Users\user\PycharmProjects\subclass_segformer"
project_maskformer_path = r"C:\Users\user\PycharmProjects\MaskFormer"
project_unet_path = r"C:\Users\user\PycharmProjects\unet"
project_sam3_path = r"C:\Users\user\PycharmProjects\SAM3"

for path in [
    project_segformer_path,
    project_maskformer_path,
    project_unet_path,
    project_sam3_path,
]:
    if path not in sys.path:
        sys.path.append(path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 統一設定對比基準為 512x512
input_tensor = torch.randn(1, 3, 512, 512).to(device)

maskformer_model_path = "facebook/maskformer-swin-tiny-ade"
mask2former_model_path = "facebook/mask2former-swin-tiny-coco-panoptic"
id2label = {0: "background", 1: "target"}

# ==========================================
# 2. 開始測試各專案模型
# ==========================================

# --- 💡 新增：Proposed Baseline (SubclassSegFormer_Unified) ---
# --- 💡 改良後的 Proposed Baseline 差異分析 ---
try:
    from semseg.models.segformer import SegFormer
    from semseg.models.subclass_segformer_unified import SubclassSegFormer_Unified

    # 1. 實例化兩個模型
    model_base = SegFormer(backbone="MiT-B0", num_classes=2).to(device).eval()
    model_unified = SubclassSegFormer_Unified(
        backbone="MiT-B0", num_classes=2, subclass=64, num_prompts=5
    ).to(device).eval()

    # 2. 計算精確參數 (單位：個)
    params_base = sum(p.numel() for p in model_base.parameters())
    params_unified = sum(p.numel() for p in model_unified.parameters())
    diff_params = params_unified - params_base

    # 3. 計算精確 GFLOPs (透過 thop 獲取原始浮點數，而非 print 的字串)
    macs_base, _ = profile(model_base, inputs=(input_tensor,), verbose=False)
    macs_unified, _ = profile(model_unified, inputs=(input_tensor,), verbose=False)

    flops_base = macs_base * 2
    flops_unified = macs_unified * 2
    diff_flops = flops_unified - flops_base

    # 4. 輸出精確對比
    print("-" * 50)
    print(f"【精確對比分析】")
    print(f"SegFormer-B0 Base  : {params_base:,} params, {flops_base:,} FLOPs")
    print(f"Proposed Baseline  : {params_unified:,} params, {flops_unified:,} FLOPs")
    print(f"差異 (Difference)  : +{diff_params:,} params, +{diff_flops:,} FLOPs")
    print(f"差異佔比 (比例)    : {(diff_params/params_base)*100:.4f}%")
    print("-" * 50)

except Exception as e:
    print(f"❌ 測試失敗: {e}")


# --- SegFormer-B0 ---
try:
    from semseg.models.segformer import SegFormer

    model_seg = SegFormer(backbone="MiT-B0", num_classes=2).to(device).eval()
    macs_seg, params_seg = profile(
        model_seg, inputs=(input_tensor,), verbose=False
    )
    print(
        f"SegFormer-B0      -> Params: {params_seg/1e6:.2f}M, GFLOPs: {(macs_seg*2)/1e9:.2f}"
    )
except Exception as e:
    print(f"❌ SegFormer-B0 測試失敗: {e}")


# --- SegFormer-B1 ---
try:
    from semseg.models.segformer import SegFormer

    model_seg = SegFormer(backbone="MiT-B1", num_classes=2).to(device).eval()
    macs_seg, params_seg = profile(
        model_seg, inputs=(input_tensor,), verbose=False
    )
    print(
        f"SegFormer-B1      -> Params: {params_seg/1e6:.2f}M, GFLOPs: {(macs_seg*2)/1e9:.2f}"
    )
except Exception as e:
    print(f"❌ SegFormer-B1 測試失敗: {e}")


# --- U-Net ---
try:
    import importlib.util

    unet_file_path = r"C:\Users\user\PycharmProjects\unet\unet.py"
    spec = importlib.util.spec_from_file_location("custom_unet", unet_file_path)
    unet_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(unet_module)

    model_unet = unet_module.UNet(in_channels=3, num_classes=2).to(device).eval()
    macs_unet, params_unet = profile(
        model_unet, inputs=(input_tensor,), verbose=False
    )
    print(
        f"U-Net             -> Params: {params_unet/1e6:.2f}M, GFLOPs: {(macs_unet*2)/1e9:.2f}"
    )
except Exception as e:
    print(f"❌ U-Net 測試失敗: {e}")


# --- DeepLabV3+ ---
try:
    from torchvision.models.segmentation.deeplabv3 import (
        DeepLabHead,
        DeepLabV3,
    )
    from torchvision.models.segmentation.fcn import FCNHead

    def prepare_model(num_classes: int) -> DeepLabV3:
        deeplabv3_model: DeepLabV3 = torch.hub.load(
            "pytorch/vision:v0.10.0",
            "deeplabv3_resnet50",
            weights="DeepLabV3_ResNet50_Weights.DEFAULT",
        )
        in_features = deeplabv3_model.classifier[0].convs[0][0].in_channels
        deeplabv3_model.classifier = DeepLabHead(
            in_channels=in_features, num_classes=num_classes
        )
        aux_in_features = deeplabv3_model.aux_classifier[0].in_channels
        deeplabv3_model.aux_classifier = FCNHead(
            in_channels=aux_in_features, channels=num_classes
        )
        return deeplabv3_model

    model_dlv3 = prepare_model(num_classes=2).to(device).eval()
    macs_dl, params_dl = profile(
        model_dlv3, inputs=(input_tensor,), verbose=False
    )
    print(
        f"DeepLabV3+        -> Params: {params_dl/1e6:.2f}M, GFLOPs: {(macs_dl*2)/1e9:.2f}"
    )
except Exception as e:
    print(f"❌ DeepLabV3+ 測試失敗: {e}")


# --- MaskFormer ---
try:
    from transformers import MaskFormerForInstanceSegmentation

    model_mf = (
        MaskFormerForInstanceSegmentation.from_pretrained(
            maskformer_model_path, id2label=id2label, ignore_mismatched_sizes=True
        )
        .to(device)
        .eval()
    )

    class MFWrapper(torch.nn.Module):

        def __init__(self, model):
            super().__init__()
            self.model = model

        def forward(self, x):
            return self.model(pixel_values=x)

    wrapped_mf = MFWrapper(model_mf)
    macs_mf, params_mf = profile(
        wrapped_mf, inputs=(input_tensor,), verbose=False
    )
    print(
        f"MaskFormer        -> Params: {params_mf/1e6:.2f}M, GFLOPs: {(macs_mf*2)/1e9:.2f}"
    )
except Exception as e:
    print(f"❌ MaskFormer 測試失敗: {e}")


# --- Mask2Former ---
try:
    from transformers import Mask2FormerForUniversalSegmentation

    model_m2f = (
        Mask2FormerForUniversalSegmentation.from_pretrained(
            mask2former_model_path, id2label=id2label, ignore_mismatched_sizes=True
        )
        .to(device)
        .eval()
    )

    class M2FWrapper(torch.nn.Module):

        def __init__(self, model):
            super().__init__()
            self.model = model

        def forward(self, x):
            return self.model(pixel_values=x)

    wrapped_m2f = M2FWrapper(model_m2f)
    macs_m2f, params_m2f = profile(
        wrapped_m2f, inputs=(input_tensor,), verbose=False
    )
    print(
        f"Mask2Former       -> Params: {params_m2f/1e6:.2f}M, GFLOPs: {(macs_m2f*2)/1e9:.2f}"
    )
except Exception as e:
    print(f"❌ Mask2Former 測試失敗: {e}")

--------------------------------------------------
【精確對比分析】
SegFormer-B0 Base  : 3,730,656 params, 14,061,404,160.0 FLOPs
Proposed Baseline  : 3,736,034 params, 14,057,209,856.0 FLOPs
差異 (Difference)  : +5,378 params, +-4,194,304.0 FLOPs
差異佔比 (比例)    : 0.1442%
--------------------------------------------------
SegFormer-B0      -> Params: 3.73M, GFLOPs: 14.06
SegFormer-B1      -> Params: 13.69M, GFLOPs: 27.01
U-Net             -> Params: 31.04M, GFLOPs: 437.94


Using cache found in C:\Users\user/.cache\torch\hub\pytorch_vision_v0.10.0


DeepLabV3+        -> Params: 41.99M, GFLOPs: 347.58


Loading weights:   0%|          | 0/432 [00:00<?, ?it/s]

[transformers] MaskFormerForInstanceSegmentation LOAD REPORT from: facebook/maskformer-swin-tiny-ade
Key                    | Status   |                                                                                         
-----------------------+----------+-----------------------------------------------------------------------------------------
class_predictor.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([151]) vs model:torch.Size([3])          
class_predictor.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([151, 256]) vs model:torch.Size([3, 256])
criterion.empty_weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([151]) vs model:torch.Size([3])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


MaskFormer        -> Params: 41.67M, GFLOPs: 98.75


Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

[transformers] Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-tiny-coco-panoptic
Key                    | Status   |                                                                                         
-----------------------+----------+-----------------------------------------------------------------------------------------
class_predictor.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([134, 256]) vs model:torch.Size([3, 256])
criterion.empty_weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([134]) vs model:torch.Size([3])          
class_predictor.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([134]) vs model:torch.Size([3])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Mask2Former       -> Params: 44.95M, GFLOPs: 122.38
